In [ ]:
import glob
import os
from copy import copy
from functools import wraps
from itertools import chain
from typing import Any, Callable

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import blab_pyutils.plotting as plth
import scipy.constants as sc
from dotenv import load_dotenv
from blab_pyutils.plotting.annotations import axis_units, escape_latex
from blab_pyutils.units import *
from blab_pyutils.plotting.configs import IEEEConfig as plot_config

from pygmid.Lookup.lut import Lookup
from pygmid.Lookup.unit_lut import UnitLookup

load_dotenv()
USE_UNITS = globals().get("USE_UNITS", os.getenv("USE_UNITS", False))

lk = UnitLookup if USE_UNITS else Lookup
mpl.rcParams.update(plot_config().config_params)
legend_kwargs = dict(
    loc='center right',
    title='L',
    bbox_to_anchor=(1.3 if USE_UNITS else 1.25, 0.5),
)

FULL = True
lk_path = next(filter(lambda p: FULL ^ p.startswith("test"), chain.from_iterable(map(lambda ext: glob.glob(f"*.{ext}"), ["h5", "hdf5", "mat", "pkl"]))))
print(lk_path)


def latex_units(func: Callable) -> Callable:
    @wraps(func)
    def latex_units_wrapper(*args, **kwargs) -> Any:
        default_format = copy(UREG.formatter.default_format)
        UREG.formatter.default_format = "~L"
        out = func(*args, **kwargs).replace('[', '[$').replace(']', '$]')
        UREG.formatter.default_format = default_format
        return out
    return latex_units_wrapper

In [ ]:
for dev in ['n', 'p']:
    fet = lk(lk_path, device=dev)
    fet_str = "\n\t".join(str(fet).splitlines())
    print(f"{dev.capitalize()}:\n{fet_str}")

dev = 'n'
fet = lk(lk_path, device=dev)

def get_plot_data(fet):
    # sweep variable vectors
    l = fet['L']
    vgs = fet['VGS']
    vds = fet['VDS']
    vsb = fet['VSB']
    VDD = fet['VDD']
    if l.size > 10:
        len_vec = l[np.arange(0, l.size - 1, l.size // 10)]
    else:
        len_vec = l

    L1 = min(l)
    VDS1 = VDD/2
    VSB1 = 0 * getattr(VDS1, "units", 1)
    VDS2 = VDD*3/4
    VSB2 = 0 * getattr(VDS2, "units", 1)
    gm_id1 = fet.lookup('GM_ID', L=L1, VDS=VDS1, VSB=VSB1)
    gm_id2 = fet.lookup('GM_ID', L=len_vec, VDS=VDS1, VSB=VSB1)
    ft1 = fet.lookup('GM_CGG', L=L1, VDS=VDS1, VSB=VSB1)/2/np.pi
    ft2 = fet.lookup('GM_CGG', L=len_vec, VDS=VDS1, VSB=VSB1)/2/np.pi
    if hasattr(ft1, "units"):
        ft1 = ft1.to(UREG.gigahertz)
        ft2 = ft2.to(UREG.gigahertz)
    vt = fet.lookup('VT', L=L1, VGS=VDD*3/4, VDS=VDS1, VSB=VSB1)
    gm_gds2 = fet.lookup('GM_GDS', L=len_vec, VDS=VDS1, VSB=VSB1)
    jd2 = fet.lookup('ID_W', L=len_vec, VDS=VDS1, VSB=VSB1)
    gamma = fet.lookup('STH_GM', L=len_vec, VDS=VDS1, VSB=VSB1) / (4*sc.Boltzmann*(UREG.boltzmann_constant if USE_UNITS else 1)*(Kelvin if USE_UNITS else lambda x: x)(fet['TEMP']))
    sfl = fet.lookup('SFL', L=len_vec, VDS=VDS1, VSB=VSB1)**0.5
    sfl_gate = sfl / fet.lookup('GM', L=len_vec, VDS=VDS1, VSB=VSB1)

    return vars()

vars().update(get_plot_data(fet))

In [ ]:
def vgs_gmid_ft(fig, ax1, **kwargs):
    ax2 = ax1.twinx()
    axis_units(ax1)
    axis_units(ax2)

    # plot gm/ID and fT versus gate bias
    ax1.grid(axis='x')
    color = 'tab:blue'
    ax1.plot(vgs, gm_id1, color=color)
    ax1.set_xlabel('$V_{GS}$')
    ax1.set_ylabel('$g_m/I_D$', color=color)
    ax1.tick_params(axis='y', labelcolor=color)
    
    color = 'tab:red'
    ax2.plot(vgs, ft1, color=color)
    ax2.set_ylabel('$f_T$', color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    fig.tight_layout()
    ax.set_title(dev+', $L$='+str(L1)+', $V_{DS}$='+str(VDS1)+', $V_{SB}$='+str(VSB1))
    ax1.set_xlim(0, np.max(vgs))
    ax2.set_xlim(0, np.max(vgs))
    ax1.axvline(x=vt, color='k', linestyle='--')

fig, ax = plt.subplots()
vgs_gmid_ft(fig, ax)
plt.show()

In [ ]:
def gmid_gmidft(fig, ax, **kwargs):
    axis_units(ax)
    # plot product of gm/ID and fT versus gm/ID
    ax.plot(gm_id1, gm_id1*ft1)
    ax.set_xlabel('$g_m/I_D$')
    ax.set_ylabel(r'$f_T\cdot g_m/I_D$')
    ax.set_title(dev+', $L$='+str(L1)+', $V_{DS}$='+str(VDS1)+', $V_{SB}$='+str(VSB1))

fig, ax = plt.subplots()
gmid_gmidft(fig, ax)
plt.show()


In [ ]:
def gmid_ft(fig, ax, **kwargs):
    axis_units(ax)
    # plot fT versus gm/ID
    ax.plot(gm_id1, ft1)
    ax.set_xlabel('$g_m/I_D$')
    ax.set_ylabel('$f_T$')
    ax.set_title(dev+', $L$='+str(L1)+', $V_{DS}$='+str(VDS1)+', $V_{SB}$='+str(VSB1))
    
fig, ax = plt.subplots()
gmid_ft(fig, ax)
plt.show()

In [ ]:
def ft_gmid_l(fig, ax, show_legend=True, **kwargs):
    axis_units(ax)
    # plot fT versus gm/ID for all L
    ax.plot(gm_id2.transpose(), ft2.transpose())
    if show_legend:
        ax.legend(labels=len_vec.tolist(), **legend_kwargs)
    ax.set_xlabel('$g_m/I_D$')
    ax.set_ylabel('$f_T$')
    ax.set_title(dev+', $V_{DS}$='+str(VDS1)+', $V_{SB}$='+str(VSB1))

fig, ax = plt.subplots()
ft_gmid_l(fig, ax)
plt.show()

In [ ]:
def gmgds_gmid_l(fig, ax, show_legend=True, **kwargs):
    axis_units(ax)
    # plot gm/gds versus gm/ID for all L
    ax.plot(gm_id2.transpose(), gm_gds2.transpose())
    if show_legend:
        ax.legend(labels=len_vec.tolist(), **legend_kwargs)
    ax.set_xlabel('$g_m/I_D$')
    ax.set_ylabel('$g_m/g_{ds}$')
    ax.set_title(dev+', $V_{DS}$='+str(VDS1)+', $V_{SB}$='+str(VSB1))

fig, ax = plt.subplots()
gmgds_gmid_l(fig, ax)
plt.show()

In [ ]:
def jd_gmid_l(fig, ax, show_legend=True, **kwargs):
    axis_units(ax)
    # plot jd versus gm/ID for all L
    ax.semilogy(gm_id2.transpose(), jd2.transpose())
    if show_legend:
        ax.legend(labels=len_vec.tolist(), **legend_kwargs)
    ax.set_ylim(1e-10, None)
    ax.set_xlabel('$g_m/I_D$')
    ax.set_ylabel('$J_D$')
    ax.set_title(dev+', $V_{DS}$='+str(VDS1)+', $V_{SB}$='+str(VSB1))

fig, ax = plt.subplots()
jd_gmid_l(fig, ax)
plt.show()

In [ ]:
def gamma_gmid_l(fig, ax, show_legend=True, **kwargs):
    axis_units(ax)
    # plot gamma versus gm/ID for all L
    ax.plot(gm_id2.transpose(), gamma.transpose())
    if show_legend:
        ax.legend(labels=len_vec.tolist(), **legend_kwargs)
    ax.set_xlabel('$g_m/I_D$')
    ax.set_ylabel(r'Thermal noise factor $\gamma$')
    ax.set_title(dev+', $V_{DS}$='+str(VDS1)+', $V_{SB}$='+str(VSB1))

fig, ax = plt.subplots()
gamma_gmid_l(fig, ax)
plt.show()

In [ ]:
def flicker_gmid_l(fig, ax, show_legend=True, **kwargs):
    axis_units(ax, y_format_wrapper=latex_units)
    # plot flicker noise drain current at 1Hz versus gm/ID for all L
    ax.semilogy(gm_id2.transpose(), sfl.transpose())
    if show_legend:
        ax.legend(labels=len_vec.tolist(), **legend_kwargs)
    # plt.ylim(1e-2, 1e2)
    ax.set_xlabel('$g_m/I_D$')
    ax.set_ylabel('1/f drain noise current at 1Hz')
    ax.set_title(dev+', $V_{DS}$='+str(VDS1)+', $V_{SB}$='+str(VSB1))

fig, ax = plt.subplots()
flicker_gmid_l(fig, ax)
plt.show()

In [ ]:
def gatenoise_gmid_l(fig, ax, show_legend=True, **kwargs):
    axis_units(ax, y_format_wrapper=latex_units)
    # plot gate-referred 1/f noise at 1Hz versus gm/ID for all L
    ax.semilogy(gm_id2.transpose(), sfl_gate.transpose())
    if show_legend:
        ax.legend(labels=len_vec.tolist(), **legend_kwargs)
    ax.set_xlabel('$g_m/I_D$')
    ax.set_ylabel('gate-referred 1/f noise at 1Hz')
    ax.set_title(dev+', $V_{DS}$='+str(VDS1)+', $V_{SB}$='+str(VSB1))

fig, ax = plt.subplots()
gatenoise_gmid_l(fig, ax)
plt.show()

In [ ]:
func_order = [
    [vgs_gmid_ft, gmid_gmidft, gmid_ft],
    [ft_gmid_l, gmgds_gmid_l, jd_gmid_l],
    [gamma_gmid_l, flicker_gmid_l, gatenoise_gmid_l],
]

for dev in ['n', 'p']:
    fet = lk(lk_path, device=dev)
    vars().update(get_plot_data(fet))
    fig, ax = plt.subplots(3, 3, tight_layout=True, figsize=tuple(map(lambda x: 3*x, mpl.rcParams["figure.figsize"])))
    fig.suptitle(escape_latex(fet.__repr__())+"\n")

    for y, (ax_row, func_row) in enumerate(zip(ax, func_order)):
        for x, (ax, f) in enumerate(zip(ax_row, func_row)):
            f(fig, ax, show_legend=(x,y)==(2,1))

    plt.show()